# 06 — Publish YC model artifacts to Hugging Face

This notebook uploads the trained artifacts under `exploration-on-yc-data/artifacts/` to a Hugging Face repo.

## What gets published

- Hybrid bundle: `hybrid_cluster_bundle.joblib`
- Feature columns: `hybrid_cluster_feature_columns.json`
- Clustering objects: `hybrid_cluster_kmeans.joblib`
- Global/cluster models: `hybrid_cluster_models.joblib`
- XAI bundle: `artifacts/hybrid_xai/*` (SHAP explainers, LIME training sample, surrogate, rules, CBR index)
- Baseline metrics CSV: `baseline_model_metrics_test.csv`

## Auth

Set an environment variable `HF_TOKEN` with a Hugging Face write token before running:

```bash
export HF_TOKEN=hf_...your_token...
```


In [ ]:
%pip install -q huggingface_hub python-dotenv


In [ ]:
import os
from pathlib import Path
import json

from dotenv import load_dotenv
from huggingface_hub import HfApi

# Load .env if present (env vars still override)
HERE = Path.cwd().resolve()
ROOT = HERE.parent  # repo root (parent of exploration-on-yc-data)
load_dotenv(ROOT / ".env", override=False)
load_dotenv(HERE / ".env", override=False)

ART = HERE / "artifacts"
assert ART.exists(), f"Missing artifacts dir: {ART}"

HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Missing HF_TOKEN. Put it in .env or export it in your shell."

# Change these if you want a different repo
HF_REPO_ID = os.environ.get("HF_REPO_ID", "bhuvesh/propertylens-yc-hybrid")  # format: username/repo
HF_REPO_TYPE = os.environ.get("HF_REPO_TYPE", "model")  # 'model' or 'dataset'
HF_PRIVATE = os.environ.get("HF_PRIVATE", "true").lower() in ("1", "true", "yes")

print("Artifacts:", ART)
print("Repo:", HF_REPO_ID, "| type:", HF_REPO_TYPE, "| private:", HF_PRIVATE)


In [ ]:
api = HfApi(token=HF_TOKEN)
repo_url = api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    private=HF_PRIVATE,
    exist_ok=True,
)
print("Repo ready:", repo_url)


## Select files to upload

By default we upload the **current** hybrid artifacts (not the `explore_*` ones).
If you want to include everything, set `UPLOAD_ALL_ARTIFACTS = True`.


In [ ]:
UPLOAD_ALL_ARTIFACTS = os.environ.get("UPLOAD_ALL_ARTIFACTS", "false").lower() in ("1", "true", "yes")

allow = {
    # main hybrid
    "hybrid_cluster_bundle.joblib",
    "hybrid_cluster_models.joblib",
    "hybrid_cluster_kmeans.joblib",
    "hybrid_cluster_feature_columns.json",
    "hybrid_cluster_meta.json",
    # baseline metrics
    "baseline_model_metrics_test.csv",
    # xai
    "hybrid_xai",
}

to_upload = []
for p in ART.iterdir():
    if UPLOAD_ALL_ARTIFACTS:
        to_upload.append(p)
    else:
        if p.name in allow:
            to_upload.append(p)

print("UPLOAD_ALL_ARTIFACTS:", UPLOAD_ALL_ARTIFACTS)
print("Planned uploads:")
for p in sorted(to_upload, key=lambda x: x.name):
    if p.is_dir():
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        print(" -", p.name + "/", f"({n} files)")
    else:
        print(" -", p.name, f"({p.stat().st_size/1e6:.1f} MB)")


## Upload

This uploads each selected file/folder into the Hugging Face repo under an `artifacts/` prefix.


In [ ]:
commit_message = os.environ.get("HF_COMMIT_MESSAGE", "Publish YC hybrid artifacts")

for p in sorted(to_upload, key=lambda x: x.name):
    if p.is_dir():
        api.upload_folder(
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            folder_path=str(p),
            path_in_repo=f"artifacts/{p.name}",
            commit_message=commit_message,
            token=HF_TOKEN,
        )
        print("Uploaded folder:", p.name)
    else:
        api.upload_file(
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            path_or_fileobj=str(p),
            path_in_repo=f"artifacts/{p.name}",
            commit_message=commit_message,
            token=HF_TOKEN,
        )
        print("Uploaded file:", p.name)

print("Done. Repo:", f"https://huggingface.co/{HF_REPO_ID}")


## Optional: add a lightweight README

If you want the repo to be self-describing, create a `README.md` in the HF repo that explains:
- data provenance
- temporal split
- what each artifact file is
- how to load + run inference (`yc_hybrid_inference.py`)
